# 13. Fleet Volt-VAr conformance: single-phase and three-phase houses

**Goal.** Fleet conformance with the AS/NZS 4777.2:2020 Volt-VAr slope requirement from the net meter only, for the
three cohorts validated in notebooks 06/07, 11 and 12, reported per cohort and pooled.

| Cohort | Test | Validated in | Main population |
|---|---|---|---|
| 1L1P | single-phase test on the house circuit | 06 / 07 | all eligible sites (as 07) |
| 3L1P | single-phase test on the blind-chosen phase | 11 | clean sites (`use_in_11`), `month_ok` months |
| 3L3P | total of the three phases vs average voltage (Clause 3.3.2.1) | 12 | clean sites (`use_in_12`), `month_ok`, no CATCH Power |

**Rules.** A site-month is conformant if its slope is in [−4.000, −2.667] %/V. Site level (as 07): conformant if it
passes in **all** its assessable months; sensitivity: in at least half. Not assessable is not the same as
non-conformant: only assessable site-months count.

**Why per cohort first.** The cohorts catch conformant inverters at different rates (recall 0.834 / 0.877 / 0.713 in
notebooks 06/11/12), so a pooled rate understates conformance most where recall is lowest. Section 6 corrects each
cohort's rate for its validated error rates, as a sensitivity.

Sections: 2 inputs · 3 funnel · 4 conformance per cohort · 5 by OEM, DNSP and install year · 6 bias-adjusted rates ·
7 sensitivities · 8 outputs · 9 limitations · 10 reading the results. Library: `lib/ami_fleet_conformance.py`.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

_current = Path.cwd().resolve()
REPO_ROOT = next((p for p in (_current, *_current.parents) if (p / "bms_sa_review").is_dir()), None)
if REPO_ROOT is None:
    raise RuntimeError(f"Could not locate the CICCADA repository root from {_current}")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from bms_sa_review.ami_analysis.lib import ami_fleet_vvar as Fleet
from bms_sa_review.ami_analysis.lib import ami_phase1p_vvar as P1
from bms_sa_review.ami_analysis.lib import ami_fleet_conformance as F13

## 1. Settings

In [ ]:
SLOPE_TOLERANCE_PCT = 20                        # band [-4.000, -2.667] %/V
SITE_PASS_SHARE, SITE_PASS_SHARE_SENS = 1.0, 0.5
STANDARD_MANDATORY = pd.Timestamp("2021-12-18")  # as notebook 07
MIN_GROUP_SITES = 10                             # hide groups with fewer sites
N_BOOT, SEED = 1000, 42

V_LO, V_HI = 242, 253
FLEET_DIR = Fleet.store_dir() / "vvar_fleet" / f"2024_2025_band_{V_LO}_{V_HI}"   # notebook 07 (1L1P)
DIR_3L1P = Fleet.store_dir() / "vvar_phase" / "3l1p_vvar"                         # notebook 11
DIR_3L3P = Fleet.store_dir() / "vvar_phase" / "3l3p_vvar"                         # notebook 12
PHASE_DIR = Fleet.store_dir() / "vvar_phase" / "phase_config"                     # notebook 10
OUT_DIR = Fleet.store_dir() / "vvar_phase" / "fleet_conformance"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Outputs -> {OUT_DIR}")

## 2. Inputs

One site-month table for the three cohorts: the net-meter verdict (`passed`) and, where the PV-circuit reference is
assessable, the reference verdict (`pass_truth`, used only for the check in section 6). Site metadata adds OEM, DNSP
and install date.

In [ ]:
site_meta = pd.read_parquet(Fleet.Config.store_path("ami_site_metadata_residential"))
load = lambda variant: F13.add_metadata(
    F13.load_fleet(FLEET_DIR, DIR_3L1P, DIR_3L3P, tolerance_pct=SLOPE_TOLERANCE_PCT, variant=variant),
    site_meta, STANDARD_MANDATORY)

fleet = load("main")
sites = F13.site_level(fleet, SITE_PASS_SHARE, SITE_PASS_SHARE_SENS)
print(f"{len(fleet):,} assessable site-months at {fleet.site_id.nunique():,} sites")
display(fleet.groupby(["cohort", "year"]).agg(sites=("site_id", "nunique"), site_months=("passed", "size")).unstack())

## 3. Funnel

How many sites each cohort starts with and how many are assessable at least once. 1L1P starts from notebook 07's
eligible sites; 3L1P and 3L3P from notebook 10's reference-graded sites, then the clean selection.

In [ ]:
ev = pd.read_parquet(PHASE_DIR / "site_evidence.parquet")
elig = pd.read_parquet(FLEET_DIR / "site_eligibility.parquet")
d3 = pd.read_parquet(DIR_3L3P / "site_month_details_3l3p.parquet")
catch_sites = set(d3.loc[d3.catch, "site_id"])
funnel = pd.DataFrame({
    "1L1P": {"start": (elig.eligibility == "eligible").sum(), "clean / eligible": (elig.eligibility == "eligible").sum()},
    "3L1P": {"start": ((ev.cohort == "3L1P") & (ev.ref_config == "1P")).sum(), "clean / eligible": ev.use_in_11.sum()},
    "3L3P": {"start": ((ev.cohort == "3L3P") & (ev.ref_config == "3P")).sum(),
             "clean / eligible": (ev.use_in_12 & ~ev.site_id.isin(catch_sites)).sum()},
}).T
funnel["assessable at least once"] = sites.groupby("cohort").site_id.nunique()
funnel["assessable site-months"] = fleet.groupby("cohort").size()
funnel["% of start assessable"] = (100 * funnel["assessable at least once"] / funnel["start"]).round(1)
display(funnel)

## 4. Conformance per cohort

Site-month rate, site-level rate (all months) and the majority-rule sensitivity, per cohort and pooled, then by year.
The validation results from notebooks 06/11/12 are shown alongside: the fleet rate is only as good as the test.

In [ ]:
rates = F13.cohort_rates(fleet, sites, by=["cohort"])
display(rates.round(1))
# By year: site level within each year (a site counts in a year if it has an assessable month that year)
by_year = pd.concat({y: F13.cohort_rates(g, F13.site_level(g, SITE_PASS_SHARE, SITE_PASS_SHARE_SENS), by=["cohort"])
                     for y, g in fleet.groupby("year")}, names=["year"])
display(by_year.round(1))

validation = F13.validation_sets(FLEET_DIR, DIR_3L1P, DIR_3L3P, SLOPE_TOLERANCE_PCT)
val = F13.validation_summary(validation)
display(val[["sites", "n", "accuracy", "precision", "recall", "specificity", "f1",
             "pass_rate_tested_%", "pass_rate_reference_%"]].round(3))

## 5. By OEM, DNSP and install year (pooled, with the cohort mix)

Groups with fewer than `MIN_GROUP_SITES` sites are hidden. The `sites_*` columns show how many sites of each cohort
sit in a group, because the cohorts' recall differs (section 6).

In [ ]:
for label, col in [("OEM", "oem"), ("DNSP", "dnsp"), ("Install era", "install_era"), ("Install year", "install_year")]:
    print(f"\n=== Conformance by {label} ===")
    t = F13.rate_table(fleet, sites, col, MIN_GROUP_SITES)
    if col in ("install_year", "install_era"):
        t = t.sort_index()
    display(t.round(1))

In [ ]:
# Site-level conformance by install year, per cohort (years with >= MIN_GROUP_SITES sites in that cohort)
fig, ax = plt.subplots(figsize=(12, 3.8), dpi=150, layout="constrained")
colors = {"1L1P": "#1f4e79", "3L1P": "#D85A30", "3L3P": "#1baf7a"}
for k, (cohort, g) in enumerate(sites.dropna(subset=["install_year"]).groupby("cohort")):
    y = g.groupby("install_year").agg(n=("site_id", "size"), pct=("conformant", "mean"))
    y = y[y.n >= MIN_GROUP_SITES]
    ax.bar(y.index + (k - 1) * 0.27, 100 * y.pct, 0.27, color=colors[cohort], label=f"{cohort} ({y.n.sum():,} sites)")
first_after = STANDARD_MANDATORY.year + (1 if STANDARD_MANDATORY.month == 12 else 0)
ax.axvline(first_after - 0.5, color="#374151", ls="--", lw=1)
ax.text(first_after - 0.4, ax.get_ylim()[1] * 0.95, "AS/NZS 4777.2:2020 mandatory →", fontsize=7, va="top")
ax.set(xlabel="Installation year", ylabel="Sites conformant in all\nassessable months (%)")
ax.legend(fontsize=8)
ax.grid(axis="y", color="#ebebeb", lw=0.5)
fig.savefig(OUT_DIR / "site_conformance_by_install_year_cohort.png", bbox_inches="tight")
plt.show()

## 6. Bias-adjusted rates (sensitivity)

`p_adj = (p_obs + specificity − 1) / (sensitivity + specificity − 1)` per cohort, with sensitivity (recall) and
specificity from that cohort's validation set (Rogan and Gladen, 1978; check the reference before citing). Interval:
bootstrap over fleet sites and validation sites. **Assumption:** the validation error rates hold for the whole cohort.
Only site-month rates are adjusted (the all-months site rule has no simple correction).

**Check (out of sample).** On the validation site-months themselves the adjusted rate equals the reference rate by
construction, so that is no test. Instead, sensitivity and specificity from one year are applied to the other year's
fleet site-months that have a reference, and compared with that year's reference rate (`error_*_pp` = percentage
points off the reference).

In [ ]:
adjusted = F13.adjusted_rates(fleet, validation, N_BOOT, SEED)
display(adjusted.round(3))
cross_year = F13.adjusted_cross_year(fleet, validation)
display(cross_year.round(3))

## 7. Sensitivities

1. **All decided months** (3L1P/3L3P): every month the blind method routes, not only clean sites and good months; the
   end-to-end accuracy from notebooks 11/12 applies (0.956 / 0.923).
2. **`n_decided` ≥ 3** for the three-phase cohorts.
3. **CATCH Power included** in 3L3P.
4. **Missing paired circuit** (3L1P, notebook 11): sites that lose their paired circuit for a year vs the rest.

In [ ]:
sens = {"main": rates}
for variant in ["all_decided", "n_decided_3", "catch_included"]:
    fv = load(variant)
    sens[variant] = F13.cohort_rates(fv, F13.site_level(fv, SITE_PASS_SHARE, SITE_PASS_SHARE_SENS), by=["cohort"])
sens_table = pd.concat({k: v[["sites", "site_months", "site_month_conformant_%", "site_conformant_%"]]
                        for k, v in sens.items()}, axis=1)
display(sens_table.round(1))

In [ ]:
blind_month = pd.read_parquet(PHASE_DIR / "blind_config_site_month.parquet")
aff = P1.affected_sites(blind_month)
f31 = fleet[fleet.cohort == "3L1P"].merge(aff[["site_id", "affected_year"]], on="site_id", how="left")
f31["group"] = np.where(f31.affected_year.fillna(0) > 0, "lost paired circuit for a year", "not affected")
s31 = F13.site_level(f31.drop(columns="cohort").rename(columns={"group": "cohort"}), SITE_PASS_SHARE, SITE_PASS_SHARE_SENS)
display(F13.cohort_rates(f31.drop(columns="cohort").rename(columns={"group": "cohort"}), s31, by=["cohort"]).round(1))

## 8. Outputs

| File | Content |
|---|---|
| `fleet_site_month.parquet` | main population: cohort, site-month, slope, verdict, reference verdict where assessable, metadata |
| `fleet_site.parquet` | one row per site: months, share passed, conformant (all months / ≥ 50%) |
| `*.csv` | cohort rates, validation summary, adjusted rates, sensitivities, funnel |

In [ ]:
fleet.to_parquet(OUT_DIR / "fleet_site_month.parquet", index=False)
sites.to_parquet(OUT_DIR / "fleet_site.parquet", index=False)
funnel.to_csv(OUT_DIR / "funnel.csv")
rates.to_csv(OUT_DIR / "cohort_rates.csv")
val.to_csv(OUT_DIR / "validation_summary.csv")
adjusted.to_csv(OUT_DIR / "adjusted_rates.csv")
cross_year.to_csv(OUT_DIR / "adjusted_cross_year_check.csv")
sens_table.to_csv(OUT_DIR / "sensitivities.csv")
print(f"Saved to {OUT_DIR}")

## 9. Limitations

- **Slope only.** The test checks the slope of Q against V in 242-253 V, not whether the inverter follows the whole
  curve (offsets and shifted deadbands pass if the slope is right).
- **Validation transfer.** Error rates come from site-months where the PV circuit is assessable; the bias adjustment
  assumes they hold for the cohort.
- **Coverage.** 3L3P sites have short data blocks at the source (median 4 good months per clean site); 3L1P sites that
  lose their paired circuit for a year (notebook 11, section 8) pass less often by the reference in their intact
  year, so dropping their missing year can shift the 3L1P rate.
- **Selection.** Three-phase results use clean sites; the "all decided" sensitivity shows how much this matters.
- **Phantom export and household load.** Both bias the net-meter slope (notebooks 09, 11, 12); recall is lowest for
  3L3P because the total carries the whole house's reactive load.

## 10. Reading the results

To be written from the outputs after the run.